<h1 style="font-family:verdana;"> <center>🏆 SPR 2026 Mammography - Baseline [0.77321] 🏆</center> </h1>


***

This notebook presents a baseline for the __SPR 2026 Mammography Report Classification__ competition. 

Minimal preprocessing is used, and applied to both training and test sets. The dataset contains duplicate or near-duplicate reports, which can cause leakage if KFold is used. So I used GroupedKFold so that identical texts never appear in both training and validation folds. 

For feature extraction, I used word-level and character-level TF-IDF features. It generally works better than using either option. 

The classifier is a LinearSVC trained on sparse TF-IDF features using class_weight='balanced' to handle the heavy class imbalance this dataset shows. 

CV is performed using Macro F1, which is the metric for this competition. No tuning was performed as it's just meant to be a baseline.

This baseline achieves ~0.73 in CV, and ~0.77 on the public leaderboard. 

***

# 1. Notebook Setup

In [ ]:
# Baseline notebook: SPR 2026 Mammography Report Classification

import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline, FeatureUnion

import hashlib

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5

TRAIN_PATH = "/kaggle/input/spr-2026-mammography-report-classification/train.csv"
TEST_PATH  = "/kaggle/input/spr-2026-mammography-report-classification/test.csv"
SUB_PATH   = "/kaggle/input/spr-2026-mammography-report-classification/submission.csv"

In [ ]:
# Load data
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# Basic sanity checks
assert {"ID", "report", "target"}.issubset(train.columns), train.columns
assert {"ID", "report"}.issubset(test.columns), test.columns

print("train:", train.shape, "| test:", test.shape)
train.head()

***

# 2. Preprocessing

In [ ]:
# Minimal text cleaning
# - keep it light to avoid breaking medical phrases
# - mainly normalize whitespace / line breaks

_whitespace_re = re.compile(r"[ \t]+")
_newlines_re = re.compile(r"\n{2,}")

def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = _whitespace_re.sub(" ", s)
    s = _newlines_re.sub("\n", s)
    return s

In [ ]:
# Stable hash for GroupKFold (prevents leakage from duplicates)
def stable_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

train["text"] = train["report"].apply(clean_text)
test["text"]  = test["report"].apply(clean_text)

train["group"] = train["text"].apply(stable_hash)

# Quick check
print("Unique groups:", train["group"].nunique(), "out of", len(train))
train[["ID", "text", "target", "group"]].head()

***

# 3. CV Baseline

In [ ]:
# CV baseline: TF-IDF + LinearSVC (macro F1) with GroupKFold

X_text = train["text"].values
y = train["target"].astype(int).values
groups = train["group"].values

word_tfidf = TfidfVectorizer(
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    max_features=80000,
)

# Word- and Character-Level N-grams
tfidf = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf),
])

# LinearSVC is strong for high-dimensional sparse TF-IDF
# class_weight='balanced' is critical because macro F1 + heavy imbalance
base_clf = LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)

gkf = GroupKFold(n_splits=N_SPLITS)

fold_scores = []
oof_pred = np.empty(len(train), dtype=int)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_text, y, groups=groups), 1):
    X_tr = tfidf.fit_transform(X_text[tr_idx])
    X_va = tfidf.transform(X_text[va_idx])

    base_clf.fit(X_tr, y[tr_idx])
    pred_va = base_clf.predict(X_va)

    oof_pred[va_idx] = pred_va
    score = f1_score(y[va_idx], pred_va, average="macro")
    fold_scores.append(score)

    print(f"Fold {fold}/{N_SPLITS} | macro F1: {score:.5f}")

cv_mean = float(np.mean(fold_scores))
cv_std  = float(np.std(fold_scores))
oof_score = f1_score(y, oof_pred, average="macro")

print("\nCV mean:", f"{cv_mean:.5f}", "| CV std:", f"{cv_std:.5f}")
print("OOF macro F1:", f"{oof_score:.5f}")

# 4. Submission

In [ ]:
# Load evaluation/test set (exists in code-competition runtime)
if os.path.exists(TEST_PATH):
    test = pd.read_csv(TEST_PATH)
    print("test:", test.shape)
else:
    raise FileNotFoundError(
        f"test.csv not found at {TEST_PATH}. "
        "This is expected in some code competitions during interactive sessions; "
        "it should exist in the evaluation runtime."
    )

assert {"ID", "report", "target"}.issubset(train.columns), train.columns
assert {"ID", "report"}.issubset(test.columns), test.columns

In [ ]:
test["text"]  = test["report"].apply(clean_text)

In [ ]:
# Train final model on full data + create submission.csv

# Refit vectorizers on training data
X_train_full = tfidf.fit_transform(train["text"])
y_full = train["target"].astype(int).values

# Train final classifier
final_clf = LinearSVC(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    max_iter=10000,
)

final_clf.fit(X_train_full, y_full)

# Transform hidden test set
X_test = tfidf.transform(test["text"])

# Predict
test_pred = final_clf.predict(X_test).astype(int)

# Build submission file
submission = pd.DataFrame({
    "ID": test["ID"],
    "target": test_pred
})

# Save 
submission.to_csv("submission.csv", index=False)

print("submission.csv saved")
submission.head()